# 17. 퍼널 단계별 태그 분포 비교 및 검정

**분석 목적:** 2023~2025년 인디게임을 리뷰 규모 기준으로 두 그룹(0~9개, 10개 이상)으로 나누어,
각 그룹에서 특정 태그가 통계적으로 유의미하게 더 많이 나타나는지 확인한다.

**활용 관점:** 출시 후 초기 유저 반응을 얻지 못하는 게임과 어느 정도 반응을 얻는 게임 사이에
태그 구성 차이가 존재하는지 검증하여, 태그 선택이 초기 반응 확보에 미치는 영향을 파악한다.

In [25]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import chi2_contingency, fisher_exact
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", 50)

## 1. 데이터 로드 및 그룹 레이블 부여

`steam_indie_games_graded.csv`(리뷰 10개 이상)와 `steam_indie_games_silence.csv`(리뷰 9개 이하)를
합쳐 2023~2025년 공통 모집단을 구성하고, `total_reviews` 기준으로 두 그룹을 레이블링한다.

- `review_group = "0~9"` : total_reviews < 10 (침묵 구간)
- `review_group = "10+"` : total_reviews >= 10 (반응 획득 구간)

In [26]:
GRADED_PATH = Path("../../../data/preprocessed/steam_indie_games_graded.csv")
SILENCE_PATH = Path("../../../data/preprocessed/steam_indie_games_silence.csv")

graded = pd.read_csv(GRADED_PATH)
silence = pd.read_csv(SILENCE_PATH)

for df in [graded, silence]:
    df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
    df["release_year"] = df["release_date"].dt.year

graded = graded.query("2023 <= release_year <= 2025").copy()
silence = silence.query("2023 <= release_year <= 2025").copy()

COLS = ["appid", "name", "release_year", "total_reviews", "tags"]
base = (
    pd.concat([graded[COLS], silence[COLS]], ignore_index=True)
    .drop_duplicates(subset="appid")
)

base["total_reviews"] = pd.to_numeric(base["total_reviews"], errors="coerce").fillna(0)
base["review_group"] = base["total_reviews"].apply(lambda v: "10+" if v >= 10 else "0~9")

print(f"전체 게임 수: {len(base):,}")
print(base["review_group"].value_counts().to_string())
base.head()

전체 게임 수: 15,673
review_group
10+    8997
0~9    6676


,appid,name,release_year,total_reviews,tags,review_group
0,226620,Desktop Dungeons,2023,2276,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ...",10+
1,230210,ASYLUM,2025,348,"{""Dark"": 40, ""Gore"": 52, ""Indie"": 76, ""Gothic""...",10+
2,251570,7 Days to Die,2024,370046,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""...",10+
3,252190,Defender's Quest 2: Mists of Ruin,2025,255,"{""2D"": 70, ""RPG"": 94, ""Indie"": 59, ""Sci-fi"": 4...",10+
4,269770,Secrets of Grindea,2024,8270,"{""2D"": 513, ""RPG"": 520, ""Cute"": 495, ""JRPG"": 5...",10+


## 2. 태그 파싱 및 Explode

`tags` 컬럼은 `{"Roguelike": 260, "Turn-Based": 135, ...}` 형태의 JSON 문자열이므로
`json.loads`로 dict로 변환한 뒤 태그명만 추출해 `explode()`로 펼친다.

전체 태그를 사용하되, 전체 모집단에서 등장 게임 수가 `MIN_TAG_GAMES` 미만인 희귀 태그는
검정력이 낮아 신뢰하기 어려우므로 제외한다.

In [27]:
MIN_TAG_GAMES = 30

def parse_tags(val):
    try:
        return list(json.loads(val).keys())
    except Exception:
        return []

base["tag_list"] = base["tags"].apply(parse_tags)

exploded = base.explode("tag_list").rename(columns={"tag_list": "tag"})
exploded = exploded[exploded["tag"].notna() & (exploded["tag"] != "")].copy()

# 전체 모집단 기준 희귀 태그 제거
tag_game_counts = exploded.groupby("tag")["appid"].nunique()
valid_tags = tag_game_counts[tag_game_counts >= MIN_TAG_GAMES].index
exploded = exploded[exploded["tag"].isin(valid_tags)].copy()

print(f"explode 후 행 수: {len(exploded):,}")
print(f"분석 대상 태그 수 (게임 수 {MIN_TAG_GAMES}개 이상): {exploded['tag'].nunique()}")
print()
print(exploded["tag"].value_counts().head(10))

explode 후 행 수: 275,625
분석 대상 태그 수 (게임 수 30개 이상): 359

tag
Singleplayer    12267
Indie            9987
Casual           7435
Adventure        7055
2D               6885
Action           6787
3D               5321
Atmospheric      4103
Colorful         3987
Exploration      3962
Name: count, dtype: int64


## 3. 그룹별 태그 보유 비율 집계

각 그룹(0~9, 10+) 안에서 해당 태그를 보유한 게임의 비율을 계산한다.

분모는 그룹별 전체 고유 게임 수, 분자는 해당 태그를 가진 고유 게임 수(`nunique`)로 집계해
한 게임이 여러 태그에 중복 집계되는 구조를 의도적으로 허용한다.

In [28]:
group_totals = base.groupby("review_group")["appid"].nunique().rename("total_games")

tag_counts = (
    exploded.groupby(["review_group", "tag"])["appid"]
    .nunique()
    .reset_index(name="game_count")
)

tag_ratio = tag_counts.merge(group_totals, on="review_group")
tag_ratio["ratio"] = tag_ratio["game_count"] / tag_ratio["total_games"] * 100

# 두 그룹 모두에 존재하는 태그만 유지
tag_pivot = tag_ratio.pivot(index="tag", columns="review_group", values="ratio").dropna()
tag_pivot.columns.name = None
tag_pivot = tag_pivot.rename(columns={"0~9": "ratio_0_9", "10+": "ratio_10plus"})
tag_pivot["ratio_diff"] = tag_pivot["ratio_10plus"] - tag_pivot["ratio_0_9"]
tag_pivot = tag_pivot.sort_values("ratio_diff", ascending=False)

print(f"분석 대상 태그 수: {len(tag_pivot)}")
tag_pivot.round(2).head(10)

분석 대상 태그 수: 354


,ratio_0_9,ratio_10plus,ratio_diff
tag,,,
Story Rich,14.60,24.96,10.36
Simulation,17.80,25.50,7.70
Atmospheric,21.87,29.38,7.51
Adventure,40.74,48.18,7.44
Exploration,21.17,28.33,7.17
Psychological Horror,8.61,15.41,6.79
RPG,16.37,23.01,6.64
Horror,13.71,19.44,5.73
First-Person,17.39,23.10,5.71


## 4. 시각화: 그룹별 태그 보유 비율 비교 (상위/하위 20개)

태그 수가 많으므로 비율 차이(`ratio_diff`) 기준 상위 20개(10+ 우세)와 하위 20개(0~9 우세)만 시각화한다.

In [29]:
TOP_N = 10
top_tags = tag_pivot.head(TOP_N).index.tolist()
bottom_tags = tag_pivot.tail(TOP_N).index.tolist()
display_tags = bottom_tags + top_tags

plot_df = tag_ratio[tag_ratio["tag"].isin(display_tags)].copy()
plot_df["review_group"] = pd.Categorical(plot_df["review_group"], categories=["0~9", "10+"], ordered=True)

tag_order = tag_pivot.loc[display_tags].sort_values("ratio_diff").index.tolist()

fig = px.bar(
    plot_df,
    x="ratio",
    y="tag",
    color="review_group",
    barmode="group",
    orientation="h",
    title=f"그룹별 태그 보유 비율 비교 (2023~2025, ratio_diff 상위·하위 각 {TOP_N}개)",
    labels={"ratio": "태그 보유 비율(%)", "tag": "태그", "review_group": "리뷰 그룹"},
    color_discrete_map={"0~9": "#e07a5f", "10+": "#3d405b"},
    category_orders={"tag": tag_order},
)

fig.update_layout(
    template="plotly_white",
    width=1000,
    height=max(500, len(display_tags) * 26),
    legend_title_text="리뷰 그룹",
    xaxis=dict(ticksuffix="%"),
)

fig.show()

## 5. 카이제곱 검정 및 FDR 보정

각 태그에 대해 2×2 분할표를 구성해 두 그룹 간 태그 보유 비율 차이가 통계적으로 유의미한지 검정한다.

```
                 해당 태그 보유   해당 태그 미보유
review_group=0~9      a               b
review_group=10+      c               d
```

- 기대 빈도 ≥ 5: Chi-square test
- 기대 빈도 < 5: Fisher's exact test

태그 수만큼 반복 검정하므로 Benjamini-Hochberg 방법으로 FDR 보정을 적용한다.

In [30]:
n_0_9 = group_totals["0~9"]
n_10plus = group_totals["10+"]

game_count_pivot = (
    tag_counts
    .pivot(index="tag", columns="review_group", values="game_count")
    .fillna(0)
    .rename(columns={"0~9": "count_0_9", "10+": "count_10plus"})
)

results = []
for tag, row in game_count_pivot.iterrows():
    a = int(row["count_0_9"])
    b = int(n_0_9 - a)
    c = int(row["count_10plus"])
    d = int(n_10plus - c)

    table = [[a, b], [c, d]]
    expected = chi2_contingency(table, correction=False).expected_freq

    if (expected < 5).any():
        _, p = fisher_exact(table)
        test_method = "fisher"
    else:
        _, p, _, _ = chi2_contingency(table, correction=False)
        test_method = "chi2"

    results.append({
        "tag": tag,
        "count_0_9": a,
        "count_10plus": c,
        "p_value": p,
        "test_method": test_method,
    })

test_df = pd.DataFrame(results)

reject, p_adjusted, _, _ = multipletests(test_df["p_value"], method="fdr_bh")
test_df["p_adjusted"] = p_adjusted
test_df["significant"] = reject

test_df = test_df.merge(tag_pivot.reset_index(), on="tag")
test_df = test_df.sort_values("p_adjusted")

print(f"유의미한 태그 수 (FDR < 0.05): {test_df['significant'].sum()}")
test_df[test_df["significant"]].head(20).round(4)

유의미한 태그 수 (FDR < 0.05): 192


,tag,count_0_9,count_10plus,p_value,test_method,p_adjusted,significant,ratio_0_9,ratio_10plus,ratio_diff
292,Story Rich,975,2246,0.0,chi2,0.0,True,14.6046,24.9639,10.3593
239,Psychological Horror,575,1386,0.0,chi2,0.0,True,8.6129,15.4051,6.7922
26,Arcade,1452,1281,0.0,chi2,0.0,True,21.7496,14.2381,-7.5115
54,Casual,3522,3913,0.0,chi2,0.0,True,52.7561,43.4923,-9.2639
274,Simulation,1188,2294,0.0,chi2,0.0,True,17.7951,25.4974,7.7023
155,Indie,4589,5398,0.0,chi2,0.0,True,68.7388,59.9978,-8.7410
25,Anime,405,1004,0.0,chi2,0.0,True,6.0665,11.1593,5.0928
32,Atmospheric,1460,2643,0.0,chi2,0.0,True,21.8694,29.3765,7.5071
245,RPG,1093,2070,0.0,chi2,0.0,True,16.3721,23.0077,6.6356
111,Exploration,1413,2549,0.0,chi2,0.0,True,21.1654,28.3317,7.1663


## 6. Odds Ratio 및 95% 신뢰구간

통계적으로 유의미한 장르에 대해 효과 크기를 계산한다.

- OR > 1 : 0~9 그룹에서 해당 장르 비율이 더 높음 → 침묵 위험 장르
- OR < 1 : 10+ 그룹에서 해당 장르 비율이 더 높음 → 반응 획득 장르

In [31]:
def odds_ratio_ci(a, b, c, d):
    # Haldane-Anscombe 보정 (0 셀 방지)
    a, b, c, d = a + 0.5, b + 0.5, c + 0.5, d + 0.5
    or_val = (a * d) / (b * c)
    log_or = np.log(or_val)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    ci_lower = np.exp(log_or - 1.96 * se)
    ci_upper = np.exp(log_or + 1.96 * se)
    return or_val, ci_lower, ci_upper

sig_df = test_df[test_df["significant"]].copy()

or_rows = []
for _, row in sig_df.iterrows():
    a = int(row["count_0_9"])
    b = int(n_0_9 - a)
    c = int(row["count_10plus"])
    d = int(n_10plus - c)
    or_val, ci_lo, ci_hi = odds_ratio_ci(a, b, c, d)
    or_rows.append({"tag": row["tag"], "odds_ratio": or_val, "ci_lower": ci_lo, "ci_upper": ci_hi})

or_df = pd.DataFrame(or_rows).merge(sig_df[["tag", "ratio_0_9", "ratio_10plus", "p_adjusted"]], on="tag")
or_df["direction"] = or_df["odds_ratio"].apply(lambda v: "침묵 위험 (0~9 우세)" if v > 1 else "반응 획득 (10+ 우세)")
or_df = or_df.sort_values("odds_ratio", ascending=False)

print(f"침묵 위험 태그: {(or_df['direction'] == '침묵 위험 (0~9 우세)').sum()}개")
print(f"반응 획득 태그: {(or_df['direction'] == '반응 획득 (10+ 우세)').sum()}개")
or_df.round(3).head(20)

침묵 위험 태그: 59개
반응 획득 태그: 133개


,tag,odds_ratio,ci_lower,ci_upper,ratio_0_9,ratio_10plus,p_adjusted,direction
103,Software,2.737,1.650,4.540,0.674,0.245,0.000,침묵 위험 (0~9 우세)
172,MOBA,2.431,1.216,4.856,0.330,0.133,0.018,침묵 위험 (0~9 우세)
66,Match 3,2.362,1.727,3.229,1.618,0.689,0.000,침묵 위험 (0~9 우세)
138,Time Attack,1.986,1.317,2.995,0.839,0.422,0.002,침묵 위험 (0~9 우세)
139,Battle Royale,1.955,1.308,2.922,0.869,0.445,0.002,침묵 위험 (0~9 우세)
173,Spelling,1.912,1.162,3.148,0.554,0.289,0.019,침묵 위험 (0~9 우세)
49,Runner,1.826,1.504,2.217,3.655,2.034,0.000,침묵 위험 (0~9 우세)
104,3D Vision,1.818,1.358,2.436,1.588,0.878,0.000,침묵 위험 (0~9 우세)
29,Score Attack,1.758,1.530,2.020,7.160,4.201,0.000,침묵 위험 (0~9 우세)
188,Trivia,1.714,1.089,2.699,0.629,0.367,0.034,침묵 위험 (0~9 우세)


## 7. 시각화: Forest Plot

유의미한 태그의 Odds Ratio와 95% 신뢰구간을 Forest Plot으로 시각화한다.
태그 수가 많을 수 있으므로 OR 기준 상위 20개(침묵 위험)와 하위 20개(반응 획득)만 표시한다.
기준선(OR=1)을 중심으로 오른쪽은 침묵 위험 태그, 왼쪽은 반응 획득 태그를 나타낸다.

In [32]:
FOREST_TOP_N = 10
forest_df = pd.concat([
    or_df[or_df["direction"] == "침묵 위험 (0~9 우세)"].tail(FOREST_TOP_N),
    or_df[or_df["direction"] == "반응 획득 (10+ 우세)"].head(FOREST_TOP_N),
]).sort_values("odds_ratio")

color_map = {"침묵 위험 (0~9 우세)": "#e07a5f", "반응 획득 (10+ 우세)": "#3d405b"}

fig = go.Figure()

for _, row in forest_df.iterrows():
    color = color_map[row["direction"]]
    fig.add_trace(go.Scatter(
        x=[row["ci_lower"], row["odds_ratio"], row["ci_upper"]],
        y=[row["tag"]] * 3,
        mode="lines+markers",
        marker=dict(size=[4, 10, 4], color=color, symbol=["line-ns", "circle", "line-ns"]),
        line=dict(color=color, width=2),
        showlegend=False,
        hovertemplate=(
            f"<b>{row['tag']}</b><br>"
            f"OR: {row['odds_ratio']:.3f}<br>"
            f"95% CI: [{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]<br>"
            f"p_adjusted: {row['p_adjusted']:.4f}<br>"
            f"0~9 비율: {row['ratio_0_9']:.1f}%<br>"
            f"10+ 비율: {row['ratio_10plus']:.1f}%<br>"
            "<extra></extra>"
        ),
    ))

for label, color in color_map.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None],
        mode="markers",
        marker=dict(size=10, color=color),
        name=label,
    ))

fig.add_vline(x=1, line_dash="dash", line_color="gray", line_width=1.5)

fig.update_layout(
    template="plotly_white",
    title=f"태그별 Odds Ratio — 침묵 위험 / 반응 획득 각 상위 {FOREST_TOP_N}개",
    xaxis=dict(title="Odds Ratio (log scale)", type="log"),
    yaxis=dict(title=""),
    width=900,
    height=max(500, len(forest_df) * 26),
    legend=dict(title="방향"),
)

fig.show()

## 8. 단계별 태그 비율 추이

앞선 검정에서 유의미하게 나온 태그들이 G0(0개) → G1(1~9개) → G2(10~49개) → G3(50~499개) → G4(500개 이상)
각 단계를 거치면서 비율이 어떻게 변하는지 확인한다.

침묵 위험 태그는 초반 단계에서 비율이 높다가 낮아지는 패턴을, 반응 획득 태그는 단계가 올라갈수록 비율이 높아지는 패턴을 보일 것으로 예상한다.

In [33]:
MILESTONE_ORDER = ["G0 (0개)", "G1 (1~9개)", "G2 (10~49개)", "G3 (50~499개)", "G4 (500개+)"]

def assign_milestone(v):
    if v == 0:       return "G0 (0개)"
    elif v <= 9:     return "G1 (1~9개)"
    elif v <= 49:    return "G2 (10~49개)"
    elif v <= 499:   return "G3 (50~499개)"
    else:            return "G4 (500개+)"

base["milestone"] = base["total_reviews"].apply(assign_milestone)

milestone_totals = base.groupby("milestone")["appid"].nunique().rename("total_games")

milestone_tag = (
    exploded.merge(base[["appid", "milestone"]], on="appid", how="left")
    .groupby(["milestone", "tag"])["appid"]
    .nunique()
    .reset_index(name="game_count")
    .merge(milestone_totals, on="milestone")
)
milestone_tag["ratio"] = milestone_tag["game_count"] / milestone_tag["total_games"] * 100

print("단계별 게임 수:")
print(milestone_totals.reindex(MILESTONE_ORDER).to_string())

단계별 게임 수:
milestone
G0 (0개)          138
G1 (1~9개)       6538
G2 (10~49개)     4904
G3 (50~499개)    2981
G4 (500개+)      1112


In [34]:
TREND_TOP_N = 10
silence_tags = or_df[or_df["direction"] == "침묵 위험 (0~9 우세)"].head(TREND_TOP_N)["tag"].tolist()
response_tags = or_df[or_df["direction"] == "반응 획득 (10+ 우세)"].head(TREND_TOP_N)["tag"].tolist()
trend_tags = silence_tags + response_tags

trend_df = (
    milestone_tag[milestone_tag["tag"].isin(trend_tags)]
    .assign(milestone=lambda d: pd.Categorical(d["milestone"], categories=MILESTONE_ORDER, ordered=True))
    .sort_values(["tag", "milestone"])
)

# 방향 레이블 부여
direction_map = {t: "침묵 위험 (0~9 우세)" for t in silence_tags}
direction_map.update({t: "반응 획득 (10+ 우세)" for t in response_tags})
trend_df["direction"] = trend_df["tag"].map(direction_map)

color_map = {"침묵 위험 (0~9 우세)": "#e07a5f", "반응 획득 (10+ 우세)": "#3d405b"}

fig = px.line(
    trend_df,
    x="milestone",
    y="ratio",
    color="tag",
    line_dash="direction",
    facet_col="direction",
    markers=True,
    title=f"유의미한 태그의 마일스톤별 비율 추이 (각 방향 상위 {TREND_TOP_N}개)",
    labels={"milestone": "리뷰 단계", "ratio": "태그 보유 비율(%)", "tag": "태그"},
    category_orders={"milestone": MILESTONE_ORDER},
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.update_traces(line_width=2, marker_size=6)
fig.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    hovermode="x unified",
    yaxis=dict(ticksuffix="%"),
    yaxis2=dict(ticksuffix="%"),
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.show()

## 9. 종합 해석

분석 결과를 두 방향으로 정리한다.

- **침묵 위험 태그 (OR > 1)**: 0~9 리뷰 그룹에서 비율이 통계적으로 유의미하게 높은 태그.
  마일스톤 추이에서도 G0~G1 구간에 집중되어 있고 G3 이상으로 갈수록 비율이 낮아지는 패턴이 확인된다면, 해당 태그가 초기 반응 확보에 불리하다는 해석을 강화한다.

- **반응 획득 태그 (OR < 1)**: 10+ 리뷰 그룹에서 비율이 통계적으로 유의미하게 높은 태그.
  단계가 올라갈수록 비율이 높아지는 패턴이라면, 이 태그를 가진 게임이 초기 반응을 넘어 더 높은 단계까지 도달하는 경향이 있다는 의미다.

태그 자체가 성공을 보장하지는 않는다. 이 결과는 초기 반응 확보 가능성에 대한 경향성이며,
가격·장르 조합·출시 전 위시리스트 확보 등 다른 요인과 함께 해석해야 한다.